# 02 Multi Sample Inference v0.2

**Notebook version:** v0.2  
**Category:** inference  
**Purpose:** Select one ROI-FCN run, one distance-orientation regression run, and one raw synthetic corpus, then execute either a multi-sample inference pass or a brightness-sensitivity sweep over a corpus slice.  
**Inputs:** `./models`, `./input`, `./src/inference_v0_1`  
**Expected outputs:** tabular inference summaries for the requested corpus slice, aggregate brightness-sensitivity summaries, per-sample drift rankings, and optional JSON save artifacts for standard inference.  
**Artifact write mode:** optional aggregated JSON under `./output/<model-name>/inference-output_<model-name>.json` for standard multi-sample inference


In [4]:
# Repo Setup
from pathlib import Path
import sys


INFERENCE_ROOT_CANDIDATES = ('05_inference-v0.2', '05_inference-v0.1', '04_inference-v0.1')


def find_repo_root(start: Path | None = None) -> tuple[Path, Path]:
    candidate = (start or Path.cwd()).resolve()
    if candidate.is_file():
        candidate = candidate.parent

    for current in (candidate, *candidate.parents):
        for name in INFERENCE_ROOT_CANDIDATES:
            inference_root = current / name
            if (inference_root / 'src').exists() and (inference_root / 'models').exists():
                return current, inference_root

    raise RuntimeError(
        f'Could not locate repo root containing any of: {INFERENCE_ROOT_CANDIDATES}'
    )


repo_root, inference_root = find_repo_root()
src_root = inference_root / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print(f'repo_root={repo_root}')
print(f'inference_root={inference_root}')


repo_root=/home/mitch/development/raccoon-ball
inference_root=/home/mitch/development/raccoon-ball/05_inference-v0.2


In [5]:
# Control Pane
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

from inference_v0_1 import (
    default_raw_corpus_roots,
    discover_model_runs,
    discover_raw_corpora,
    load_corpus_samples,
    run_multi_sample_inference,
)

all_models = discover_model_runs(inference_root / 'models')
if not all_models:
    raise FileNotFoundError(f'No model run artifacts found under {inference_root / "models"}')

distance_models = [artifact for artifact in all_models if artifact.model_family == 'distance-orientation']
roi_models = [artifact for artifact in all_models if artifact.model_family == 'roi-fcn']
if not distance_models:
    raise FileNotFoundError('No distance-orientation model runs found under ./models/distance-orientation')
if not roi_models:
    raise FileNotFoundError('No ROI-FCN model runs found under ./models/roi-fcn')

raw_corpus_roots = default_raw_corpus_roots()
corpora = discover_raw_corpora()
if not corpora:
    searched_roots = '\n  - '.join(str(path) for path in raw_corpus_roots)
    raise FileNotFoundError(
        'No valid raw-image corpora found in any configured raw-corpus root:\n'
        f'  - {searched_roots}\n'
        'Expected corpus folders with images/ plus manifests/run.json and manifests/samples.csv.'
    )

corpus_names = [corpus.name for corpus in corpora]
duplicate_corpus_names = sorted({name for name in corpus_names if corpus_names.count(name) > 1})
if duplicate_corpus_names:
    raise ValueError(
        f'Duplicate corpus names found across raw-corpus roots: {duplicate_corpus_names}'
    )

distance_model_by_label = {artifact.label: artifact for artifact in distance_models}
roi_model_by_label = {artifact.label: artifact for artifact in roi_models}
corpus_by_name = {corpus.name: corpus for corpus in corpora}
corpus_samples_cache: dict[str, pd.DataFrame] = {}
local_input_root = inference_root / 'input'
all_input_dirs = sorted(path.name for path in local_input_root.iterdir() if path.is_dir()) if local_input_root.exists() else []
ignored_dirs = sorted(set(all_input_dirs).difference(corpus_by_name))


def selected_model_output_dir_name(model_artifact) -> str:
    run_dir = Path(model_artifact.run_dir).resolve()
    if run_dir.name.startswith('run_') and run_dir.parent.name == 'runs' and run_dir.parent.parent.name:
        return run_dir.parent.parent.name
    return run_dir.name


print(f'discovered_models_total={len(all_models)}')
print(f'discovered_distance_orientation_models={len(distance_models)}')
print(f'discovered_roi_fcn_models={len(roi_models)}')
print(f'discovered_raw_corpora={len(corpora)}')
print('raw_corpus_roots=', [str(path) for path in raw_corpus_roots if path.exists()])
if ignored_dirs:
    print('ignored_non_raw_input_dirs=', ignored_dirs)


def get_corpus_samples(corpus_name: str) -> pd.DataFrame:
    if corpus_name not in corpus_samples_cache:
        corpus_samples_cache[corpus_name] = load_corpus_samples(corpus_by_name[corpus_name])
    return corpus_samples_cache[corpus_name]


distance_model_select = widgets.Select(
    options=[artifact.label for artifact in distance_models],
    value=distance_models[0].label,
    description='Regressor',
    rows=min(8, max(4, len(distance_models))),
)
roi_model_select = widgets.Select(
    options=[artifact.label for artifact in roi_models],
    value=roi_models[-1].label,
    description='ROI-FCN',
    rows=min(8, max(4, len(roi_models))),
)
corpus_select = widgets.Select(
    options=[corpus.name for corpus in corpora],
    value=corpora[0].name,
    description='Corpus',
    rows=min(8, max(4, len(corpora))),
)
image_select = widgets.SelectMultiple(
    options=[],
    value=(),
    description='Image',
    rows=12,
    layout=widgets.Layout(height='280px'),
)
sample_count_input = widgets.IntText(
    value=1,
    description='# of Samples',
)
offset_input = widgets.IntText(
    value=0,
    description='Offset',
)
run_button = widgets.Button(
    description='Run Inference',
    button_style='primary',
    disabled=True,
)
save_toggle = widgets.Checkbox(value=False, description='Save JSON')
save_roi_images_toggle = widgets.Checkbox(value=False, description='Save ROI Images')
corpus_count_html = widgets.HTML()
selection_summary_html = widgets.HTML()
status_html = widgets.HTML()
results_out = widgets.Output()


def clear_results() -> None:
    status_html.value = ''
    with results_out:
        clear_output()


def update_run_state() -> None:
    corpus_name = corpus_select.value
    if not corpus_name:
        corpus_count_html.value = ''
        selection_summary_html.value = ''
        run_button.disabled = True
        return

    samples_df = get_corpus_samples(corpus_name)
    total_samples = int(len(samples_df))
    corpus_count_html.value = f'<b>Total samples in corpus:</b> {total_samples}'

    offset_value = int(offset_input.value)
    sample_count_value = int(sample_count_input.value)

    if total_samples <= 0:
        selection_summary_html.value = '<span style="color:#b00020;"><b>No selectable samples are available in this corpus.</b></span>'
        run_button.disabled = True
        return
    if offset_value < 0:
        selection_summary_html.value = '<span style="color:#b00020;"><b>Offset must be 0 or greater.</b></span>'
        run_button.disabled = True
        return
    if sample_count_value <= 0:
        selection_summary_html.value = '<span style="color:#b00020;"><b># of Samples must be greater than 0.</b></span>'
        run_button.disabled = True
        return
    if offset_value >= total_samples:
        selection_summary_html.value = (
            f'<span style="color:#b00020;"><b>Offset is out of range.</b> '
            f'Choose a value between 0 and {total_samples - 1}.</span>'
        )
        run_button.disabled = True
        return

    actual_count = min(sample_count_value, total_samples - offset_value)
    range_end = offset_value + actual_count - 1
    truncation_note = ''
    if actual_count < sample_count_value:
        truncation_note = (
            f' Requested {sample_count_value}; only {actual_count} sample(s) are available '
            f'from offset {offset_value}.'
        )

    selection_summary_html.value = (
        f'<b>Run slice:</b> {actual_count} sample(s), indices {offset_value} to {range_end}. '
        '<i>Image selection is displayed for reference only and is ignored when running inference.</i>'
        f'{truncation_note}'
    )
    run_button.disabled = False


def refresh_corpus_options(*_args) -> None:
    clear_results()
    corpus_name = corpus_select.value
    if not corpus_name:
        image_select.options = []
        image_select.value = ()
        update_run_state()
        return

    samples_df = get_corpus_samples(corpus_name)
    image_select.options = samples_df['__image_name__'].tolist()
    image_select.value = ()
    update_run_state()


def on_model_change(change) -> None:
    if change['name'] == 'value':
        clear_results()


def on_corpus_change(change) -> None:
    if change['name'] == 'value':
        refresh_corpus_options()


def on_numeric_change(change) -> None:
    if change['name'] != 'value':
        return
    if change['new'] != change['old']:
        clear_results()
    update_run_state()


def on_image_change(change) -> None:
    if change['name'] != 'value':
        return
    if change['new'] != change['old']:
        clear_results()
    update_run_state()


def on_run_clicked(_button) -> None:
    clear_results()
    selected_distance_model = distance_model_by_label[distance_model_select.value]
    selected_roi_model = roi_model_by_label[roi_model_select.value]
    selected_corpus = corpus_by_name[corpus_select.value]
    samples_df = get_corpus_samples(selected_corpus.name)
    total_samples = int(len(samples_df))
    offset_value = int(offset_input.value)
    sample_count_value = int(sample_count_input.value)

    if total_samples <= 0:
        status_html.value = '<span style="color:#b00020;"><b>No selectable samples are available in the selected corpus.</b></span>'
        return
    if offset_value < 0:
        status_html.value = '<span style="color:#b00020;"><b>Offset must be 0 or greater.</b></span>'
        return
    if sample_count_value <= 0:
        status_html.value = '<span style="color:#b00020;"><b># of Samples must be greater than 0.</b></span>'
        return
    if offset_value >= total_samples:
        status_html.value = (
            f'<span style="color:#b00020;"><b>Offset is out of range.</b> '
            f'Choose a value between 0 and {total_samples - 1}.</span>'
        )
        return

    requested_count = sample_count_value
    actual_count = min(requested_count, total_samples - offset_value)
    save_result = bool(save_toggle.value)
    save_roi_images = bool(save_roi_images_toggle.value) if save_result else False
    results_root_path = None
    if save_result:
        results_root_path = inference_root / 'output' / selected_model_output_dir_name(selected_distance_model)

    run_button.disabled = True
    status_html.value = f'<i>Running inference for {actual_count} sample(s)...</i>'
    try:
        results = run_multi_sample_inference(
            selected_distance_model.run_dir,
            selected_corpus.root,
            roi_model_run_dir=selected_roi_model.run_dir,
            offset=offset_value,
            num_samples=requested_count,
            save_result=save_result,
            save_roi_images=save_roi_images,
            results_root_path=results_root_path,
            device='cuda',
        )
    except Exception as exc:
        status_html.value = (
            f'<span style="color:#b00020;"><b>Inference failed:</b> {exc}</span>'
        )
    else:
        processed_count = len(results)
        truncation_note = ''
        if processed_count < requested_count:
            truncation_note = (
                f' Requested {requested_count}; processed {processed_count} available sample(s).'
            )
        status_html.value = (
            f'<b>Inference completed.</b> Processed {processed_count} sample(s) starting at offset {offset_value}.'
            f'{truncation_note}'
        )

        detail_rows = [
            {
                'sample_index': offset_value + index,
                'image': result.selected_image_name,
                'sample_id': result.sample_id,
                'predicted_distance_m': result.predicted_distance_m,
                'actual_distance_m': result.actual_distance_m,
                'absolute_distance_error_m': result.absolute_distance_error_m,
                'predicted_orientation_deg': result.predicted_orientation_deg,
                'actual_orientation_deg': result.actual_orientation_deg,
                'absolute_orientation_error_deg': result.absolute_orientation_error_deg,
            }
            for index, result in enumerate(results)
        ]
        detail_df = pd.DataFrame(detail_rows)
        summary_rows = [
            {'field': 'Distance model', 'value': results[0].selected_model_label},
            {'field': 'ROI-FCN model', 'value': results[0].selected_roi_model_label},
            {'field': 'Corpus', 'value': results[0].selected_corpus_name},
            {'field': 'Offset', 'value': offset_value},
            {'field': 'Requested samples', 'value': requested_count},
            {'field': 'Processed samples', 'value': processed_count},
            {
                'field': 'Mean absolute distance error (m)',
                'value': f'{detail_df["absolute_distance_error_m"].mean():.4f}',
            },
            {
                'field': 'Mean absolute orientation error (deg)',
                'value': f'{detail_df["absolute_orientation_error_deg"].mean():.4f}',
            },
            {
                'field': 'Max absolute distance error (m)',
                'value': f'{detail_df["absolute_distance_error_m"].max():.4f}',
            },
            {
                'field': 'Max absolute orientation error (deg)',
                'value': f'{detail_df["absolute_orientation_error_deg"].max():.4f}',
            },
        ]

        with results_out:
            clear_output()
            display(pd.DataFrame(summary_rows))
            display(
                detail_df.round(
                    {
                        'predicted_distance_m': 4,
                        'actual_distance_m': 4,
                        'absolute_distance_error_m': 4,
                        'predicted_orientation_deg': 4,
                        'actual_orientation_deg': 4,
                        'absolute_orientation_error_deg': 4,
                    }
                )
            )
            if save_result and results and results[0].saved_json_path is not None:
                print(f'Saved JSON: {results[0].saved_json_path}')
                if results[0].saved_roi_path is not None:
                    print(f'Saved ROI images: {processed_count}')
                else:
                    print('Saved ROI images: disabled')
    finally:
        update_run_state()


distance_model_select.observe(on_model_change, names='value')
roi_model_select.observe(on_model_change, names='value')
corpus_select.observe(on_corpus_change, names='value')
sample_count_input.observe(on_numeric_change, names='value')
offset_input.observe(on_numeric_change, names='value')
image_select.observe(on_image_change, names='value')
run_button.on_click(on_run_clicked)

refresh_corpus_options()

controls = widgets.VBox(
    [
        widgets.HBox([distance_model_select, roi_model_select, corpus_select, image_select]),
        widgets.HBox([sample_count_input, offset_input, run_button, save_toggle, save_roi_images_toggle]),
        corpus_count_html,
        selection_summary_html,
        status_html,
        results_out,
    ]
)
display(controls)


discovered_models_total=4
discovered_distance_orientation_models=1
discovered_roi_fcn_models=3
discovered_raw_corpora=1
raw_corpus_roots= ['/home/mitch/development/raccoon-ball/05_inference-v0.2/input', '/home/mitch/development/raccoon-ball/02_synthetic-data-processing-v3.0/input-images']
ignored_non_raw_input_dirs= ['26-04-11_v021-validate-shuffled-images']


## Brightness Sensitivity Investigation

This pane runs a controlled sweep over the final grayscale ROI tensor seen by the distance-orientation regressor. 
If you explicitly select images, the analysis uses those exact images; otherwise it uses the `Offset` and `# of Samples` slice. 
The sweep reports prediction drift versus the `darkness_gain=1.0` baseline, plus aggregate and per-sample summaries.


In [6]:
# Brightness Sensitivity Control Pane
import json
from datetime import datetime

import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

from inference_v0_1 import run_brightness_sensitivity_analysis

required_globals = (
    'distance_models',
    'roi_models',
    'corpora',
    'corpus_by_name',
    'get_corpus_samples',
    'selected_model_output_dir_name',
)
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        'Run the multi-sample control pane cell above before using brightness analysis. '
        f'Missing globals: {missing_globals}'
    )


def parse_darkness_gains(raw_text: str) -> tuple[float, ...]:
    text = str(raw_text).replace('\n', ',')
    tokens: list[str] = []
    for part in text.split(','):
        stripped = part.strip()
        if not stripped:
            continue
        tokens.extend(piece for piece in stripped.split() if piece)
    if not tokens:
        raise ValueError('Provide at least one darkness gain value.')

    values = tuple(float(token) for token in tokens)
    for value in values:
        if value <= 0.0:
            raise ValueError(f'Darkness gains must be > 0. Got {value}.')
    return values


def dataframe_records_for_json(df: pd.DataFrame) -> list[dict[str, object]]:
    return df.astype(object).where(pd.notna(df), None).to_dict(orient='records')


def build_brightness_analysis_output_path(model_artifact) -> Path:
    output_dir = inference_root / 'output' / selected_model_output_dir_name(model_artifact)
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d-%H%M%S-%f')
    return output_dir / f'brightness-analysis_{timestamp}.json'


analysis_distance_model_select = widgets.Select(
    options=[artifact.label for artifact in distance_models],
    value=distance_models[0].label,
    description='Regressor',
    rows=min(8, max(4, len(distance_models))),
)
analysis_roi_model_select = widgets.Select(
    options=[artifact.label for artifact in roi_models],
    value=roi_models[-1].label,
    description='ROI-FCN',
    rows=min(8, max(4, len(roi_models))),
)
analysis_corpus_select = widgets.Select(
    options=[corpus.name for corpus in corpora],
    value=corpora[0].name,
    description='Corpus',
    rows=min(8, max(4, len(corpora))),
)
analysis_image_select = widgets.SelectMultiple(
    options=[],
    value=(),
    description='Images',
    rows=12,
    layout=widgets.Layout(height='280px'),
)
analysis_sample_count_input = widgets.IntText(
    value=16,
    description='# of Samples',
)
analysis_offset_input = widgets.IntText(
    value=0,
    description='Offset',
)
analysis_gain_text = widgets.Text(
    value='0.6, 0.8, 1.0, 1.2, 1.4',
    description='Darkness gains',
    layout=widgets.Layout(width='360px'),
)
analysis_run_button = widgets.Button(
    description='Run Brightness Analysis',
    button_style='warning',
    disabled=True,
)
analysis_save_toggle = widgets.Checkbox(value=False, description='Save Analysis JSON')
analysis_help_html = widgets.HTML(
    value=(
        '<i>Tip:</i> explicit image selections override the offset / sample-count slice for brightness analysis. '
        'If no images are selected, the slice is used. When save is enabled, each run writes a new timestamped JSON artifact '
        'under the selected distance model output folder in <code>./output</code>. Progress updates refresh the single status line '
        'roughly every 250 processed samples.'
    )
)
analysis_corpus_count_html = widgets.HTML()
analysis_selection_summary_html = widgets.HTML()
analysis_status_html = widgets.HTML()
analysis_results_out = widgets.Output()
analysis_progress_interval_samples = 250


def clear_analysis_results() -> None:
    analysis_status_html.value = ''
    with analysis_results_out:
        clear_output()


def update_analysis_state() -> None:
    corpus_name = analysis_corpus_select.value
    if not corpus_name:
        analysis_corpus_count_html.value = ''
        analysis_selection_summary_html.value = ''
        analysis_run_button.disabled = True
        return

    samples_df = get_corpus_samples(corpus_name)
    total_samples = int(len(samples_df))
    analysis_corpus_count_html.value = f'<b>Total samples in corpus:</b> {total_samples}'

    try:
        parsed_gains = parse_darkness_gains(analysis_gain_text.value)
    except Exception as exc:
        analysis_selection_summary_html.value = (
            f'<span style="color:#b00020;"><b>Darkness gains are invalid.</b> {exc}</span>'
        )
        analysis_run_button.disabled = True
        return

    if total_samples <= 0:
        analysis_selection_summary_html.value = (
            '<span style="color:#b00020;"><b>No selectable samples are available in this corpus.</b></span>'
        )
        analysis_run_button.disabled = True
        return

    selected_images = list(analysis_image_select.value)
    if selected_images:
        analysis_selection_summary_html.value = (
            f'<b>Analysis selection:</b> {len(selected_images)} explicit image(s). '
            f'Offset and # of Samples will be ignored. '
            f'<b>Darkness gains:</b> {", ".join(f"{gain:g}" for gain in parsed_gains)}'
        )
        analysis_run_button.disabled = False
        return

    offset_value = int(analysis_offset_input.value)
    sample_count_value = int(analysis_sample_count_input.value)
    if offset_value < 0:
        analysis_selection_summary_html.value = (
            '<span style="color:#b00020;"><b>Offset must be 0 or greater.</b></span>'
        )
        analysis_run_button.disabled = True
        return
    if sample_count_value <= 0:
        analysis_selection_summary_html.value = (
            '<span style="color:#b00020;"><b># of Samples must be greater than 0.</b></span>'
        )
        analysis_run_button.disabled = True
        return
    if offset_value >= total_samples:
        analysis_selection_summary_html.value = (
            f'<span style="color:#b00020;"><b>Offset is out of range.</b> '
            f'Choose a value between 0 and {total_samples - 1}.</span>'
        )
        analysis_run_button.disabled = True
        return

    actual_count = min(sample_count_value, total_samples - offset_value)
    range_end = offset_value + actual_count - 1
    truncation_note = ''
    if actual_count < sample_count_value:
        truncation_note = (
            f' Requested {sample_count_value}; only {actual_count} sample(s) are available '
            f'from offset {offset_value}.'
        )
    analysis_selection_summary_html.value = (
        f'<b>Analysis slice:</b> {actual_count} sample(s), indices {offset_value} to {range_end}. '
        f'<b>Darkness gains:</b> {", ".join(f"{gain:g}" for gain in parsed_gains)}.'
        f'{truncation_note}'
    )
    analysis_run_button.disabled = False


def refresh_analysis_corpus_options(*_args) -> None:
    clear_analysis_results()
    corpus_name = analysis_corpus_select.value
    if not corpus_name:
        analysis_image_select.options = []
        analysis_image_select.value = ()
        update_analysis_state()
        return

    samples_df = get_corpus_samples(corpus_name)
    analysis_image_select.options = samples_df['__image_name__'].tolist()
    analysis_image_select.value = ()
    update_analysis_state()


def on_analysis_model_change(change) -> None:
    if change['name'] == 'value':
        clear_analysis_results()


def on_analysis_corpus_change(change) -> None:
    if change['name'] == 'value':
        refresh_analysis_corpus_options()


def on_analysis_numeric_change(change) -> None:
    if change['name'] != 'value':
        return
    if change['new'] != change['old']:
        clear_analysis_results()
    update_analysis_state()


def on_analysis_image_change(change) -> None:
    if change['name'] != 'value':
        return
    if change['new'] != change['old']:
        clear_analysis_results()
    update_analysis_state()


def on_analysis_run_clicked(_button) -> None:
    clear_analysis_results()

    selected_distance_model = distance_model_by_label[analysis_distance_model_select.value]
    selected_roi_model = roi_model_by_label[analysis_roi_model_select.value]
    selected_corpus = corpus_by_name[analysis_corpus_select.value]
    selected_samples_df = get_corpus_samples(selected_corpus.name)
    selected_images = list(analysis_image_select.value)

    try:
        parsed_gains = parse_darkness_gains(analysis_gain_text.value)
    except Exception as exc:
        analysis_status_html.value = (
            f'<span style="color:#b00020;"><b>Brightness analysis failed:</b> {exc}</span>'
        )
        return

    offset_value = int(analysis_offset_input.value)
    requested_count = int(analysis_sample_count_input.value)
    save_analysis = bool(analysis_save_toggle.value)
    analysis_run_button.disabled = True

    total_available = int(len(selected_samples_df))
    expected_total = (
        len(selected_images)
        if selected_images
        else min(requested_count, max(0, total_available - offset_value))
    )
    next_progress_update = analysis_progress_interval_samples

    if selected_images:
        run_target_text = f'{len(selected_images)} explicit image(s)'
    else:
        run_target_text = f'slice offset {offset_value}, count {requested_count}'
    analysis_status_html.value = (
        f'<i>Running brightness analysis for {run_target_text}...</i> '
        f'Processed 0/{expected_total} samples.'
    )

    def on_analysis_progress(processed_samples: int, total_samples: int) -> None:
        nonlocal next_progress_update
        if processed_samples < total_samples and processed_samples < next_progress_update:
            return
        while next_progress_update <= processed_samples:
            next_progress_update += analysis_progress_interval_samples
        analysis_status_html.value = (
            f'<i>Running brightness analysis for {run_target_text}...</i> '
            f'Processed {processed_samples}/{total_samples} samples.'
        )

    try:
        analysis = run_brightness_sensitivity_analysis(
            selected_distance_model.run_dir,
            selected_corpus.root,
            roi_model_run_dir=selected_roi_model.run_dir,
            image_names=selected_images or None,
            offset=offset_value,
            num_samples=requested_count,
            darkness_gains=parsed_gains,
            progress_callback=on_analysis_progress,
            device='cuda',
        )
    except Exception as exc:
        analysis_status_html.value = (
            f'<span style="color:#b00020;"><b>Brightness analysis failed:</b> {exc}</span>'
        )
    else:
        aggregate_df = analysis.aggregate_summary.copy()
        per_sample_df = analysis.per_sample_summary.copy()
        correlation_df = analysis.brightness_correlations.copy()
        prediction_df = analysis.predictions.copy()

        baseline_row = aggregate_df.loc[aggregate_df['darkness_gain'].sub(1.0).abs().idxmin()]
        worst_distance_row = aggregate_df.loc[aggregate_df['mean_abs_distance_shift_m'].idxmax()]
        worst_orientation_row = aggregate_df.loc[aggregate_df['mean_abs_orientation_shift_deg'].idxmax()]

        summary_rows = [
            {'field': 'Distance model', 'value': analysis.model_label},
            {'field': 'ROI-FCN model', 'value': analysis.roi_model_label},
            {'field': 'Corpus', 'value': analysis.corpus_name},
            {'field': 'Selection mode', 'value': 'explicit images' if selected_images else 'slice'},
            {'field': 'Selected images', 'value': len(selected_images)},
            {'field': 'Requested samples', 'value': requested_count if not selected_images else 'n/a'},
            {'field': 'Prediction rows', 'value': len(prediction_df)},
            {'field': 'Save analysis JSON', 'value': save_analysis},
            {
                'field': 'Darkness gains',
                'value': ', '.join(f'{gain:g}' for gain in analysis.darkness_gains),
            },
            {
                'field': 'Baseline mean abs distance error (m)',
                'value': f"{float(baseline_row['mean_abs_distance_error_m']):.4f}",
            },
            {
                'field': 'Baseline mean abs orientation error (deg)',
                'value': f"{float(baseline_row['mean_abs_orientation_error_deg']):.4f}",
            },
            {
                'field': 'Worst mean abs distance shift gain',
                'value': f"{float(worst_distance_row['darkness_gain']):g} -> {float(worst_distance_row['mean_abs_distance_shift_m']):.4f} m",
            },
            {
                'field': 'Worst mean abs orientation shift gain',
                'value': f"{float(worst_orientation_row['darkness_gain']):g} -> {float(worst_orientation_row['mean_abs_orientation_shift_deg']):.4f} deg",
            },
        ]

        analysis_artifact_path = None
        analysis_artifact_error = ''
        if save_analysis:
            payload = {
                'created_local': datetime.now().isoformat(timespec='seconds'),
                'selected_model': {
                    'label': analysis.model_label,
                    'run_dir': str(Path(selected_distance_model.run_dir).resolve()),
                },
                'selected_roi_model': {
                    'label': analysis.roi_model_label,
                    'run_dir': str(Path(selected_roi_model.run_dir).resolve()),
                },
                'selected_corpus': {
                    'name': analysis.corpus_name,
                    'root': str(Path(selected_corpus.root).resolve()),
                },
                'selection': {
                    'mode': 'explicit_images' if selected_images else 'slice',
                    'selected_images': selected_images,
                    'offset': None if selected_images else offset_value,
                    'requested_samples': None if selected_images else requested_count,
                    'prediction_rows': int(len(prediction_df)),
                },
                'darkness_gains': [float(gain) for gain in analysis.darkness_gains],
                'summary_rows': summary_rows,
                'aggregate_summary': dataframe_records_for_json(aggregate_df),
                'per_sample_summary': dataframe_records_for_json(per_sample_df),
                'brightness_correlations': dataframe_records_for_json(correlation_df),
                'predictions': dataframe_records_for_json(prediction_df),
            }
            try:
                analysis_artifact_path = build_brightness_analysis_output_path(selected_distance_model)
                analysis_artifact_path.write_text(
                    json.dumps(payload, indent=2, ensure_ascii=False),
                    encoding='utf-8',
                )
            except Exception as exc:
                analysis_artifact_error = str(exc)
            else:
                summary_rows.append(
                    {
                        'field': 'Analysis JSON artifact',
                        'value': str(analysis_artifact_path),
                    }
                )
        if save_analysis and analysis_artifact_path is None:
            summary_rows.append(
                {
                    'field': 'Analysis JSON artifact',
                    'value': f'FAILED: {analysis_artifact_error}',
                }
            )

        aggregate_display = aggregate_df.round(
            {
                'darkness_gain': 3,
                'mean_variant_vehicle_mean_darkness': 4,
                'mean_abs_distance_error_m': 4,
                'median_abs_distance_error_m': 4,
                'p95_abs_distance_error_m': 4,
                'mean_abs_distance_shift_m': 4,
                'p95_abs_distance_shift_m': 4,
                'max_abs_distance_shift_m': 4,
                'mean_abs_orientation_error_deg': 4,
                'median_abs_orientation_error_deg': 4,
                'p95_abs_orientation_error_deg': 4,
                'mean_abs_orientation_shift_deg': 4,
                'p95_abs_orientation_shift_deg': 4,
                'max_abs_orientation_shift_deg': 4,
                'delta_mean_abs_distance_error_m_vs_baseline': 4,
                'delta_mean_abs_orientation_error_deg_vs_baseline': 4,
                'delta_mean_abs_distance_shift_m_vs_baseline': 4,
                'delta_mean_abs_orientation_shift_deg_vs_baseline': 4,
            }
        )
        per_sample_display = per_sample_df.round(
            {
                'truth_distance_m': 4,
                'baseline_prediction_distance_m': 4,
                'baseline_prediction_orientation_deg': 4,
                'baseline_abs_distance_error_m': 4,
                'baseline_abs_orientation_error_deg': 4,
                'baseline_canvas_mean_intensity': 4,
                'baseline_canvas_std_intensity': 4,
                'baseline_vehicle_pixel_fraction': 4,
                'baseline_vehicle_mean_intensity': 4,
                'baseline_vehicle_std_intensity': 4,
                'baseline_vehicle_mean_darkness': 4,
                'distance_prediction_range_m': 4,
                'mean_abs_distance_shift_m': 4,
                'max_abs_distance_shift_m': 4,
                'distance_shift_slope_m_per_gain': 4,
                'mean_abs_orientation_shift_deg': 4,
                'max_abs_orientation_shift_deg': 4,
                'orientation_shift_slope_deg_per_gain': 4,
                'distance_error_range_m': 4,
                'orientation_error_range_deg': 4,
                'max_abs_distance_error_m': 4,
                'max_abs_orientation_error_deg': 4,
            }
        )
        correlation_display = correlation_df.copy()
        if not correlation_display.empty:
            correlation_display = correlation_display.round(
                {'pearson_r': 4, 'abs_pearson_r': 4}
            )
        prediction_display = prediction_df.round(
            {
                'darkness_gain': 3,
                'truth_distance_m': 4,
                'truth_orientation_deg': 4,
                'prediction_distance_m': 4,
                'prediction_orientation_deg': 4,
                'distance_error_m': 4,
                'abs_distance_error_m': 4,
                'orientation_error_deg': 4,
                'abs_orientation_error_deg': 4,
                'baseline_prediction_distance_m': 4,
                'baseline_prediction_orientation_deg': 4,
                'distance_shift_from_baseline_m': 4,
                'abs_distance_shift_from_baseline_m': 4,
                'orientation_shift_from_baseline_deg': 4,
                'abs_orientation_shift_from_baseline_deg': 4,
                'variant_canvas_mean_intensity': 4,
                'variant_canvas_std_intensity': 4,
                'variant_vehicle_pixel_fraction': 4,
                'variant_vehicle_mean_intensity': 4,
                'variant_vehicle_std_intensity': 4,
                'variant_vehicle_mean_darkness': 4,
            }
        )

        top_distance_cols = [
            'analysis_sample_index',
            'image_filename',
            'sample_id',
            'truth_distance_m',
            'baseline_vehicle_mean_darkness',
            'baseline_abs_distance_error_m',
            'max_abs_distance_shift_m',
            'distance_shift_slope_m_per_gain',
        ]
        top_orientation_cols = [
            'analysis_sample_index',
            'image_filename',
            'sample_id',
            'truth_orientation_deg',
            'baseline_vehicle_mean_darkness',
            'baseline_abs_orientation_error_deg',
            'max_abs_orientation_shift_deg',
            'orientation_shift_slope_deg_per_gain',
        ]

        analysis_status_html.value = (
            f'<b>Brightness analysis completed.</b> '
            f'Processed {int(aggregate_df["sample_count"].max())} sample(s) across '
            f'{len(analysis.darkness_gains)} darkness gain setting(s).'
        )

        with analysis_results_out:
            clear_output()
            display(pd.DataFrame(summary_rows))

            print('Aggregate summary by darkness gain')
            display(aggregate_display)

            print('Most brightness-sensitive samples by distance drift')
            display(
                per_sample_display.sort_values(
                    'max_abs_distance_shift_m',
                    ascending=False,
                    kind='stable',
                )[top_distance_cols].head(20)
            )

            print('Most brightness-sensitive samples by orientation drift')
            display(
                per_sample_display.sort_values(
                    'max_abs_orientation_shift_deg',
                    ascending=False,
                    kind='stable',
                )[top_orientation_cols].head(20)
            )

            print('Strongest baseline brightness correlations')
            if correlation_display.empty:
                print('No finite correlations were available for this run.')
            else:
                display(
                    correlation_display.loc[
                        correlation_display['abs_pearson_r'].notna()
                    ].head(20)
                )

            print(f'Prediction detail rows: {len(prediction_display)}')
            if len(prediction_display) > 100:
                print('Showing the first 100 prediction rows only.')
            display(prediction_display.head(100))

            if save_analysis:
                if analysis_artifact_path is not None:
                    print(f'Saved analysis JSON: {analysis_artifact_path}')
                else:
                    print(f'Analysis JSON save failed: {analysis_artifact_error}')
    finally:
        update_analysis_state()


analysis_distance_model_select.observe(on_analysis_model_change, names='value')
analysis_roi_model_select.observe(on_analysis_model_change, names='value')
analysis_corpus_select.observe(on_analysis_corpus_change, names='value')
analysis_sample_count_input.observe(on_analysis_numeric_change, names='value')
analysis_offset_input.observe(on_analysis_numeric_change, names='value')
analysis_gain_text.observe(on_analysis_numeric_change, names='value')
analysis_image_select.observe(on_analysis_image_change, names='value')
analysis_run_button.on_click(on_analysis_run_clicked)

refresh_analysis_corpus_options()

analysis_controls = widgets.VBox(
    [
        widgets.HBox(
            [
                analysis_distance_model_select,
                analysis_roi_model_select,
                analysis_corpus_select,
                analysis_image_select,
            ]
        ),
        widgets.HBox(
            [
                analysis_sample_count_input,
                analysis_offset_input,
                analysis_gain_text,
                analysis_run_button,
                analysis_save_toggle,
            ]
        ),
        analysis_help_html,
        analysis_corpus_count_html,
        analysis_selection_summary_html,
        analysis_status_html,
        analysis_results_out,
    ]
)
display(analysis_controls)
